- Razvan (`Databricks-Dolly/` folder)
- Mihnea (`ablation_base_flan_t5_large_config (1).json`, `ablation_flan_t5_large_student_v1_self_instruct_config.json`)

Metrics used:
- Similarity (`similarity_mean`),  Cosine similarity between the embedding of the baseline (unablated) model output and the ablated model output, computed using the Qwen3-Embedding model. This is the primary metric: it captures whether the ablated model preserves the meaning of the original response, even if the wording changes. A score of 1.0 means semantically identical outputs; lower scores indicate semantic degradation.
- ROUGE-L (`rougeL`),  Longest common subsequence overlap between baseline and ablated outputs. Measures lexical (word-level) similarity.
- METEOR (`meteor_mean`),  Token-level alignment metric with synonym/stem matching. Complements ROUGE-L by accounting for paraphrases.

All three metrics are computed against the unablated model's own output (not against a gold reference), so they measure behavioral drift caused by the ablation, not absolute task quality. All plots show Δ (delta) values relative to the unablated baseline,  negative values indicate degradation caused by the ablation.


In [ ]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 120,
    'savefig.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Data Loading

In [ ]:
BASE_DIR = '.'  # current directory
DOLLY_DIR = os.path.join(BASE_DIR, 'Databricks-Dolly')

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

# --- Flan-T5-Large (base) on Self-Instruct ---
flan_base_raw = load_json(os.path.join(BASE_DIR, 'ablation_base_flan_t5_large_config.json'))
flan_base = flan_base_raw['results']

# --- Flan-T5-Large (student v1) on Self-Instruct ---
flan_student_raw = load_json(os.path.join(BASE_DIR, 'ablation_flan_t5_large_student_v1_self_instruct_config.json'))
flan_student = flan_student_raw['results']

# --- T5-Large (pre-trained) on Dolly-15k ---
dolly_baseline = load_json(os.path.join(DOLLY_DIR, 't5_large_baseline_pretrained_dolly.json'))
dolly_component = load_json(os.path.join(DOLLY_DIR, 'component_dropping_evaluation (t5-large pre-trained dolly15k).json'))
dolly_odd_even = load_json(os.path.join(DOLLY_DIR, 'component_dropping_odd_even_evaluation (t5-large pre-trained dolly).json'))
dolly_ff_drop = load_json(os.path.join(DOLLY_DIR, 'feed_forward_layer_dropping_evaluation (t5-large pretrained dolly).json'))
dolly_attn_mask = load_json(os.path.join(DOLLY_DIR, 'attention_masking_evaluation (t5-large pretrained dolly).json'))
dolly_ff_mask = load_json(os.path.join(DOLLY_DIR, 'feed_forward_encoder_decoder_masking_evaluation (t5-large pretrained dolly).json'))
dolly_gauss = load_json(os.path.join(DOLLY_DIR, 'gaussian_noising_component_evaluation (t5-large pretrained dolly).json'))

print('All data loaded successfully.')
print(f'Flan-T5 base keys: {len(flan_base)}')
print(f'Flan-T5 student keys: {len(flan_student)}')
print(f'Dolly component dropping keys: {len(dolly_component)}')

In [ ]:
# Helper: extract metrics from a results dict
def extract_metrics(results_dict, keys, baseline_key=None):
    rows = []
    for k in keys:
        if k not in results_dict:
            continue
        v = results_dict[k]
        row = {
            'key': k,
            'similarity_mean': v['similarity_mean'],
            'rougeL': v['rouge']['rougeL'],
            'meteor_mean': v['meteor_mean'],
        }
        rows.append(row)
    df = pd.DataFrame(rows)
    if baseline_key and baseline_key in results_dict:
        bl = results_dict[baseline_key]
        df['delta_sim'] = df['similarity_mean'] - bl['similarity_mean']
        df['delta_rougeL'] = df['rougeL'] - bl['rouge']['rougeL']
        df['delta_meteor'] = df['meteor_mean'] - bl['meteor_mean']
    return df

# Dolly baseline values
dolly_bl_sim = dolly_baseline['similarity_mean']
dolly_bl_rougeL = dolly_baseline['rouge']['rougeL']
dolly_bl_meteor = dolly_baseline['meteor_mean']
print(f'Dolly baseline --- sim: {dolly_bl_sim:.4f}, rougeL: {dolly_bl_rougeL:.4f}, meteor: {dolly_bl_meteor:.4f}')

# Flan baselines
flan_base_bl = flan_base['baseline_no_drop']
flan_student_bl = flan_student['baseline_no_drop']
print(f'Flan-T5 base baseline --- sim: {flan_base_bl["similarity_mean"]:.4f}, rougeL: {flan_base_bl["rouge"]["rougeL"]:.4f}, meteor: {flan_base_bl["meteor_mean"]:.4f}')
print(f'Flan-T5 student baseline --- sim: {flan_student_bl["similarity_mean"]:.4f}, rougeL: {flan_student_bl["rouge"]["rougeL"]:.4f}, meteor: {flan_student_bl["meteor_mean"]:.4f}')

---
## Part 1: Layer / Feed-Forward Dropping

### 1.1 Single Encoder Layer Dropping
> Razvan,  T5-Large pre-trained / Dolly-15k

In [ ]:
# --- T5-Large on Dolly: single encoder block drop ---
enc_keys = [f'drop_encoder_block_{i}' for i in range(24)]
enc_data = []
for k in enc_keys:
    if k in dolly_component:
        v = dolly_component[k]
        enc_data.append({
            'layer': int(k.split('_')[-1]),
            'similarity_mean': v['similarity_mean'],
            'rougeL': v['rouge']['rougeL'],
            'meteor_mean': v['meteor_mean'],
            'delta_sim': v['similarity_mean'] - dolly_bl_sim,
            'delta_rougeL': v['rouge']['rougeL'] - dolly_bl_rougeL,
            'delta_meteor': v['meteor_mean'] - dolly_bl_meteor,
        })
df_enc_dolly = pd.DataFrame(enc_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, label in zip(axes, ['delta_sim', 'delta_rougeL', 'delta_meteor'],
                              ['Δ Similarity', 'Δ ROUGE-L', 'Δ METEOR']):
    ax.bar(df_enc_dolly['layer'], df_enc_dolly[metric], color='steelblue', alpha=0.8)
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Encoder Layer Index')
    ax.set_ylabel(label)
    ax.set_title(f'{label} --- Single Encoder Drop')
fig.suptitle('T5-Large / Dolly --- Single Encoder Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Razvan, T5-Large / Dolly): The encoder is highly redundant: 15 out of 24 layers actually improve similarity when removed (mean Δ sim = +0.001). Only encoder layer 0 causes a meaningful drop (Δ sim = −0.029); all other layers degrade by less than −0.006. Layers 23, 10, and 1 show the largest positive Δ (+0.011, +0.011, +0.010), suggesting they may introduce slight noise. The overall encoder redundancy is remarkable, any single layer can be safely removed with negligible impact.

In [ ]:
# Inspect T5 encoder drop data
print("T5-Large / Dolly, Single encoder drop, sorted by Δ Similarity (worst first):")
print(df_enc_dolly.sort_values('delta_sim')[['layer','delta_sim','delta_rougeL','delta_meteor']].to_string(index=False))
print(f"\nBaseline: sim={dolly_bl_sim:.4f}, rougeL={dolly_bl_rougeL:.4f}, meteor={dolly_bl_meteor:.4f}")
print(f"Mean Δ sim: {df_enc_dolly['delta_sim'].mean():.4f}")
print(f"Std Δ sim: {df_enc_dolly['delta_sim'].std():.4f}")
print(f"Layers with positive Δ sim (improved): {df_enc_dolly[df_enc_dolly['delta_sim'] > 0]['layer'].tolist()}")

> Mihnea,  Flan-T5-Large (base) / Self-Instruct

In [ ]:
# Same for Flan-T5 base
enc_flan = []
for i in range(24):
    k = f'drop_encoder_block_{i}'
    if k in flan_base:
        v = flan_base[k]
        enc_flan.append({
            'layer': i,
            'delta_sim': v['similarity_mean'] - flan_base_bl['similarity_mean'],
            'delta_rougeL': v['rouge']['rougeL'] - flan_base_bl['rouge']['rougeL'],
            'delta_meteor': v['meteor_mean'] - flan_base_bl['meteor_mean'],
        })
df_enc_flan = pd.DataFrame(enc_flan)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, label in zip(axes, ['delta_sim', 'delta_rougeL', 'delta_meteor'],
                              ['Δ Similarity', 'Δ ROUGE-L', 'Δ METEOR']):
    ax.bar(df_enc_flan['layer'], df_enc_flan[metric], color='darkorange', alpha=0.8)
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Encoder Layer Index')
    ax.set_ylabel(label)
    ax.set_title(f'{label} --- Single Encoder Drop')
fig.suptitle('Flan-T5-Large (base) / Self-Instruct --- Single Encoder Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Mihnea, Flan-T5 base / Self-Instruct): Similar redundancy pattern: 14 out of 24 layers improve similarity when removed (mean Δ sim ≈ 0.000). However, the distribution differs from T5-Large. Here layer 23 is the clear worst (Δ sim = −0.041), unlike T5 where layer 23 was beneficial. Layer 0, which was the worst in T5 (Δ = −0.029), is essentially harmless here (Δ = +0.001). Instruction-tuning appears to shift encoder sensitivity from early to late layers. The best layers to remove are 7 (+0.011), 6 (+0.010), and 8 (+0.008), middle layers are the most redundant.

In [ ]:
# Inspect Flan-T5 encoder drop data
print("Flan-T5 base / Self-Instruct, Single encoder drop, sorted by Δ Similarity (worst first):")
print(df_enc_flan.sort_values('delta_sim')[['layer','delta_sim','delta_rougeL','delta_meteor']].to_string(index=False))
print(f"\nBaseline: sim={flan_base_bl['similarity_mean']:.4f}")
print(f"Mean Δ sim: {df_enc_flan['delta_sim'].mean():.4f}")
print(f"Layers with positive Δ sim (improved): {df_enc_flan[df_enc_flan['delta_sim'] > 0]['layer'].tolist()}")

### 1.2 Single Decoder Layer Dropping

In [ ]:
# T5-Large on Dolly: single decoder block drop
dec_data = []
for i in range(24):
    k = f'drop_decoder_block_{i}'
    if k in dolly_component:
        v = dolly_component[k]
        dec_data.append({
            'layer': i,
            'delta_sim': v['similarity_mean'] - dolly_bl_sim,
            'delta_rougeL': v['rouge']['rougeL'] - dolly_bl_rougeL,
            'delta_meteor': v['meteor_mean'] - dolly_bl_meteor,
        })
df_dec_dolly = pd.DataFrame(dec_data)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, label in zip(axes, ['delta_sim', 'delta_rougeL', 'delta_meteor'],
                              ['Δ Similarity', 'Δ ROUGE-L', 'Δ METEOR']):
    ax.bar(df_dec_dolly['layer'], df_dec_dolly[metric], color='indianred', alpha=0.8)
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Decoder Layer Index')
    ax.set_ylabel(label)
    ax.set_title(f'{label} --- Single Decoder Drop')
fig.suptitle('T5-Large / Dolly --- Single Decoder Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Razvan,  T5-Large / Dolly): Decoder layer 0 is the single most critical layer,  dropping it causes the largest similarity drop (Δ sim ≈ −0.137), a far greater impact than any single encoder layer. Every decoder block attends to the encoder output, but block 0 is the entry point of the decoder stack: it establishes the initial decoder representation that all later blocks (and their cross-attention) build on, so removing it is the hardest to recover from via residual connections. Other decoder layers show only mild sensitivity (Δ sim ≈ −0.001 to −0.006).


In [ ]:
# Flan-T5 base: single decoder block drop
dec_flan = []
for i in range(24):
    k = f'drop_decoder_block_{i}'
    if k in flan_base:
        v = flan_base[k]
        dec_flan.append({
            'layer': i,
            'delta_sim': v['similarity_mean'] - flan_base_bl['similarity_mean'],
            'delta_rougeL': v['rouge']['rougeL'] - flan_base_bl['rouge']['rougeL'],
            'delta_meteor': v['meteor_mean'] - flan_base_bl['meteor_mean'],
        })
df_dec_flan = pd.DataFrame(dec_flan)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric, label in zip(axes, ['delta_sim', 'delta_rougeL', 'delta_meteor'],
                              ['Δ Similarity', 'Δ ROUGE-L', 'Δ METEOR']):
    ax.bar(df_dec_flan['layer'], df_dec_flan[metric], color='coral', alpha=0.8)
    ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Decoder Layer Index')
    ax.set_ylabel(label)
    ax.set_title(f'{label} --- Single Decoder Drop')
fig.suptitle('Flan-T5-Large (base) / Self-Instruct --- Single Decoder Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Mihnea,  Flan-T5 base / Self-Instruct): Unlike T5-Large where decoder block 0 was critical, Flan-T5 base shows decoder layer 6 as the single most sensitive layer (Δ sim ≈ −0.265), a much larger drop than any other layer. Decoder block 0 here has only a minor impact (Δ sim ≈ −0.005). This suggests instruction-tuning redistributes the critical cross-attention routing away from the first decoder layer and into a deeper layer, making the vulnerability profile model-specific rather than architecture-universal.

In [ ]:
# Check which decoder layer causes the biggest drop for Flan-T5 base
print("Flan-T5 base,  Decoder layers sorted by Δ Similarity (worst first):")
print(df_dec_flan.sort_values('delta_sim')[['layer','delta_sim','delta_rougeL','delta_meteor']].head(10).to_string(index=False))

### 1.3 Encoder vs Decoder Sensitivity Comparison

In [ ]:
# Overlay encoder vs decoder single-layer drop (similarity) for both models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# T5-Large / Dolly
ax = axes[0]
ax.plot(df_enc_dolly['layer'], df_enc_dolly['delta_sim'], 'o-', label='Encoder', markersize=4)
ax.plot(df_dec_dolly['layer'], df_dec_dolly['delta_sim'], 's-', label='Decoder', markersize=4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax.set_xlabel('Layer Index')
ax.set_ylabel('Δ Similarity')
ax.set_title('T5-Large / Dolly')
ax.legend()

# Flan-T5 / Self-Instruct
ax = axes[1]
ax.plot(df_enc_flan['layer'], df_enc_flan['delta_sim'], 'o-', label='Encoder', markersize=4)
ax.plot(df_dec_flan['layer'], df_dec_flan['delta_sim'], 's-', label='Decoder', markersize=4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax.set_xlabel('Layer Index')
ax.set_ylabel('Δ Similarity')
ax.set_title('Flan-T5-Large (base) / Self-Instruct')
ax.legend()

fig.suptitle('Encoder vs Decoder: Single Layer Drop Impact', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Both models): Across both T5-Large and Flan-T5, the decoder is consistently more sensitive than the encoder,  encoder Δ values cluster near zero, while decoder Δ values show clear spikes at critical layers.

Notable spikes:
- T5-Large / Dolly: Decoder layer 0 dominates with Δ sim ≈ −0.137,  by far the worst single-layer drop across both components. Encoder layer 0 is the most sensitive encoder layer (Δ sim ≈ −0.029), but still much smaller than the decoder spike.
- Flan-T5 / Self-Instruct: The spike shifts,  decoder layer 6 is the critical layer (Δ sim ≈ −0.265), not layer 0 (which only drops −0.005). On the encoder side, layer 23 is the most sensitive (Δ sim ≈ −0.041). This shift from layer 0 to layer 6 between the two models suggests instruction-tuning reorganizes which decoder layers carry the most critical cross-attention routing.

In [ ]:
# Identify the most sensitive layers in each model
print("=== T5-Large / Dolly ===")
print("Encoder,  worst 5:")
print(df_enc_dolly.sort_values('delta_sim')[['layer','delta_sim']].head(5).to_string(index=False))
print("\nDecoder,  worst 5:")
print(df_dec_dolly.sort_values('delta_sim')[['layer','delta_sim']].head(5).to_string(index=False))
print("\n=== Flan-T5 Base / Self-Instruct ===")
print("Encoder,  worst 5:")
print(df_enc_flan.sort_values('delta_sim')[['layer','delta_sim']].head(5).to_string(index=False))
print("\nDecoder,  worst 5:")
print(df_dec_flan.sort_values('delta_sim')[['layer','delta_sim']].head(5).to_string(index=False))

### 1.4 Cumulative Layer Dropping (from beginning & end)

In [ ]:
def get_cumulative_drop(results, prefix, n_layers, baseline_vals, spelling='beginning'):
    rows = []
    for i in range(n_layers):
        k = f'{prefix}_{spelling}_{i}' if 'beginning' in prefix or 'beggining' in prefix else f'{prefix}_{i}'
        # Try both spellings for Dolly
        if k not in results:
            alt_k = k.replace('beginning', 'beggining')
            if alt_k in results:
                k = alt_k
            else:
                continue
        v = results[k]
        rows.append({
            'layers_dropped': i + 1,
            'similarity_mean': v['similarity_mean'],
            'delta_sim': v['similarity_mean'] - baseline_vals[0],
            'rougeL': v['rouge']['rougeL'],
            'delta_rougeL': v['rouge']['rougeL'] - baseline_vals[1],
            'meteor_mean': v['meteor_mean'],
            'delta_meteor': v['meteor_mean'] - baseline_vals[2],
        })
    return pd.DataFrame(rows)

# T5 / Dolly --- encoder from beginning (Dolly has typo 'beggining')
dolly_bl_vals = (dolly_bl_sim, dolly_bl_rougeL, dolly_bl_meteor)

enc_beg_dolly = []
for i in range(23):
    k = f'drop_encoder_block_from_beggining_{i}'
    if k in dolly_component:
        v = dolly_component[k]
        enc_beg_dolly.append({'layers_dropped': i+1, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
df_enc_beg_dolly = pd.DataFrame(enc_beg_dolly)

enc_end_dolly = []
for i in range(23, 0, -1):
    k = f'drop_encoder_block_from_end_{i}'
    if k in dolly_component:
        v = dolly_component[k]
        enc_end_dolly.append({'layers_dropped': 24 - i, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
df_enc_end_dolly = pd.DataFrame(enc_end_dolly)

dec_beg_dolly = []
for i in range(23):
    k = f'drop_decoder_block_from_beggining_{i}'
    if k in dolly_component:
        v = dolly_component[k]
        dec_beg_dolly.append({'layers_dropped': i+1, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
df_dec_beg_dolly = pd.DataFrame(dec_beg_dolly)

dec_end_dolly = []
for i in range(23, 0, -1):
    k = f'drop_decoder_block_from_end_{i}'
    if k in dolly_component:
        v = dolly_component[k]
        dec_end_dolly.append({'layers_dropped': 24 - i, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
df_dec_end_dolly = pd.DataFrame(dec_end_dolly)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, df, title in [
    (axes[0,0], df_enc_beg_dolly, 'Encoder --- Drop from Beginning'),
    (axes[0,1], df_enc_end_dolly, 'Encoder --- Drop from End'),
    (axes[1,0], df_dec_beg_dolly, 'Decoder --- Drop from Beginning'),
    (axes[1,1], df_dec_end_dolly, 'Decoder --- Drop from End'),
]:
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    ax.plot(df['layers_dropped'], df['sim'], 'o-', label='Similarity', markersize=4)
    ax.plot(df['layers_dropped'], df['rougeL'], 's-', label='ROUGE-L', markersize=4)
    ax.plot(df['layers_dropped'], df['meteor'], '^-', label='METEOR', markersize=4)
    ax.axhline(dolly_bl_sim, color='blue', linestyle=':', alpha=0.4)
    ax.set_xlabel('Layers Dropped')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend(loc='lower left')
    ax.set_ylim(-0.05, max(dolly_bl_sim + 0.1, 0.7))

fig.suptitle('T5-Large / Dolly --- Cumulative Layer Dropping', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Findings (Razvan,  T5-Large / Dolly): Cumulative dropping from the beginning is catastrophic: removing just the first 2 encoder layers collapses all metrics to near zero (similarity ≈ 0.28, ROUGE/METEOR ≈ 0). Decoder from-beginning is even more severe,  1 layer removed already causes major damage, 2+ layers cause total collapse. In contrast, dropping from the end is much more gradual: 5-10 layers can be removed from the end before severe degradation. Later layers are more redundant than early layers.

In [ ]:
# Flan-T5 base --- cumulative layer dropping
flan_bl_vals = (flan_base_bl['similarity_mean'], flan_base_bl['rouge']['rougeL'], flan_base_bl['meteor_mean'])

def get_cumulative_flan(results, component, direction, bl_vals):
    rows = []
    for i in range(23):
        k = f'drop_{component}_block_from_{direction}_{i}'
        if k in results:
            v = results[k]
            rows.append({'layers_dropped': i+1, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
    if direction == 'end':
        for i in range(23, 0, -1):
            k = f'drop_{component}_block_from_end_{i}'
            if k in results:
                v = results[k]
                rows.append({'layers_dropped': 24-i, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
    return pd.DataFrame(rows)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
configs = [
    (axes[0,0], 'encoder', 'beginning', 'Encoder --- Drop from Beginning'),
    (axes[0,1], 'encoder', 'end', 'Encoder --- Drop from End'),
    (axes[1,0], 'decoder', 'beginning', 'Decoder --- Drop from Beginning'),
    (axes[1,1], 'decoder', 'end', 'Decoder --- Drop from End'),
]
for ax, comp, direction, title in configs:
    rows = []
    if direction == 'beginning':
        for i in range(23):
            k = f'drop_{comp}_block_from_beginning_{i}'
            if k in flan_base:
                v = flan_base[k]
                rows.append({'layers_dropped': i+1, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
    else:
        for i in range(23, 0, -1):
            k = f'drop_{comp}_block_from_end_{i}'
            if k in flan_base:
                v = flan_base[k]
                rows.append({'layers_dropped': 24-i, 'sim': v['similarity_mean'], 'rougeL': v['rouge']['rougeL'], 'meteor': v['meteor_mean']})
    df = pd.DataFrame(rows)
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    ax.plot(df['layers_dropped'], df['sim'], 'o-', label='Similarity', markersize=4)
    ax.plot(df['layers_dropped'], df['rougeL'], 's-', label='ROUGE-L', markersize=4)
    ax.plot(df['layers_dropped'], df['meteor'], '^-', label='METEOR', markersize=4)
    ax.axhline(flan_bl_vals[0], color='blue', linestyle=':', alpha=0.4)
    ax.set_xlabel('Layers Dropped')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend(loc='lower left')

fig.suptitle('Flan-T5-Large (base) / Self-Instruct --- Cumulative Layer Dropping', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

Findings (Mihnea,  Flan-T5 base / Self-Instruct): Same asymmetry observed: dropping from the beginning destroys performance rapidly, while dropping from the end allows for gradual degradation. The Flan-T5 model appears slightly more resilient to encoder pruning from the end than the pre-trained T5, possibly because instruction-tuning distributes knowledge more broadly across later layers.

### 1.5 Odd/Even Layer Dropping

In [ ]:
# Odd/Even dropping comparison --- both models
def build_odd_even_table(results, baseline_vals, label):
    odd_even_keys = [
        'drop_encoder_block_all_even', 'drop_encoder_block_all_odd',
        'drop_decoder_block_all_even', 'drop_decoder_block_all_odd',
        'drop_blocks_all_even', 'drop_blocks_all_odd',
    ]
    rows = []
    for k in odd_even_keys:
        if k in results:
            v = results[k]
            rows.append({
                'experiment': k.replace('drop_', '').replace('_block', '').replace('_all_', ' ').replace('_', ' ').title(),
                'Similarity': v['similarity_mean'],
                'ROUGE-L': v['rouge']['rougeL'],
                'METEOR': v['meteor_mean'],
                'Δ Sim': v['similarity_mean'] - baseline_vals[0],
            })
    df = pd.DataFrame(rows)
    return df

# T5/Dolly
df_oe_dolly = build_odd_even_table(dolly_odd_even, dolly_bl_vals, 'T5 Dolly')
# Flan-T5 base
df_oe_flan = build_odd_even_table(flan_base, flan_bl_vals, 'Flan base')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [(axes[0], df_oe_dolly, 'T5-Large / Dolly'), (axes[1], df_oe_flan, 'Flan-T5 (base) / Self-Instruct')]:
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    x = np.arange(len(df))
    w = 0.25
    ax.bar(x - w, df['Similarity'], w, label='Similarity')
    ax.bar(x, df['ROUGE-L'], w, label='ROUGE-L')
    ax.bar(x + w, df['METEOR'], w, label='METEOR')
    ax.set_xticks(x)
    ax.set_xticklabels(df['experiment'], rotation=35, ha='right', fontsize=8)
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend()

fig.suptitle('Odd/Even Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Both models): There is a striking odd vs even asymmetry, especially in T5-Large / Dolly:

- T5 / Dolly: Dropping all odd layers barely affects performance (Encoder Odd Δ sim = +0.005, Decoder Odd = +0.009, Blocks Odd = +0.001,  essentially identical to baseline). Dropping even layers is far more harmful, but unevenly so: the decoder and the full stack collapse (Decoder Even Δ sim = −0.214, Blocks Even = −0.288), whereas the encoder even drop is only mild (Δ sim = −0.050). So the "even layers are critical" story holds mainly for the decoder; the encoder stays redundant under either parity.
- Flan-T5 / Self-Instruct: The asymmetry persists but is less extreme. Decoder Odd is still the most survivable (Δ sim = −0.045), while Decoder Even collapses (Δ sim = −0.276). However, encoder odd/even are both severely affected (−0.171 and −0.247), unlike T5 where encoder odd was harmless.

One possible explanation is the residual structure (even layers carrying primary transforms, odd layers acting closer to pass-through), but this is speculative: it does not account for the Flan encoder, where dropping odd layers is also damaging. Treat it as a hypothesis, not a conclusion.


### 1.6 Feed-Forward Layer Dropping

In [ ]:
# FF layer dropping
ff_drop_keys = [
    'drop_feed_forward_encoder_all_even', 'drop_feed_forward_encoder_all_odd', 'drop_feed_forward_encoder_all',
    'drop_feed_forward_decoder_all_even', 'drop_feed_forward_decoder_all_odd', 'drop_feed_forward_decoder_all',
    'drop_feed_forward_all_even', 'drop_feed_forward_all_odd', 'drop_feed_forward_all',
]

def build_ff_drop_table(results, baseline_vals):
    rows = []
    for k in ff_drop_keys:
        if k in results:
            v = results[k]
            short = k.replace('drop_feed_forward_', 'FF ')
            rows.append({
                'experiment': short,
                'Similarity': v['similarity_mean'],
                'ROUGE-L': v['rouge']['rougeL'],
                'METEOR': v['meteor_mean'],
            })
    return pd.DataFrame(rows)

df_ff_dolly = build_ff_drop_table(dolly_ff_drop, dolly_bl_vals)
df_ff_flan = build_ff_drop_table(flan_base, flan_bl_vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [(axes[0], df_ff_dolly, 'T5-Large / Dolly'), (axes[1], df_ff_flan, 'Flan-T5 (base) / Self-Instruct')]:
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    x = np.arange(len(df))
    w = 0.25
    ax.bar(x - w, df['Similarity'], w, label='Similarity')
    ax.bar(x, df['ROUGE-L'], w, label='ROUGE-L')
    ax.bar(x + w, df['METEOR'], w, label='METEOR')
    ax.set_xticks(x)
    ax.set_xticklabels(df['experiment'], rotation=40, ha='right', fontsize=7)
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend()

fig.suptitle('Feed-Forward Layer Dropping', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Razvan + Mihnea): The odd/even asymmetry extends to FF layers, with notable surprises:

- T5 / Dolly: Removing even encoder FF layers actually improves similarity to 0.647 (above the 0.611 baseline), suggesting those FF sub-layers may introduce noise rather than useful signal. Removing all FF layers or all decoder FF collapses the model (sim ≈ 0.21-0.30).
- Flan-T5 / Self-Instruct: Removing odd decoder FF layers preserves similarity almost perfectly (0.653 vs 0.655 baseline), meaning half the decoder's FF parameters contribute essentially nothing. Even FF removal is far more damaging (sim ≈ 0.37-0.38).
- In both models, removing all FF layers from both encoder and decoder causes near-total collapse (sim ≈ 0.21-0.33, ROUGE-L ≈ 0), confirming FF layers collectively carry critical parametric memory.

In [ ]:
print("=== T5-Large / Dolly,  Odd/Even ===")
print(df_oe_dolly[['experiment','Similarity','ROUGE-L','METEOR','Δ Sim']].to_string(index=False))
print()
print("=== Flan-T5 Base / Self-Instruct,  Odd/Even ===")
print(df_oe_flan[['experiment','Similarity','ROUGE-L','METEOR','Δ Sim']].to_string(index=False))
print()
print("=== T5-Large / Dolly,  FF Dropping ===")
print(df_ff_dolly[['experiment','Similarity','ROUGE-L','METEOR']].to_string(index=False))
print()
print("=== Flan-T5 Base / Self-Instruct,  FF Dropping ===")
print(df_ff_flan[['experiment','Similarity','ROUGE-L','METEOR']].to_string(index=False))

---
## Part 2: Attention & Feed-Forward Masking

### 2.1 Attention Head Masking

In [ ]:
# Attention masking --- group by scope and ratio
import re

def parse_attention_masking(results, baseline_vals):
    rows = []
    for k, v in results.items():
        if not k.startswith('mask_') or 'attention' not in k:
            continue
        # Format: mask_{scope}_{attn_type}_{ratio}
        parts = k.split('_')
        ratio = float(parts[-1])
        # Find attention type and scope
        attn_type = ''
        scope = ''
        if 'self_cross_attention' in k:
            attn_type = 'self+cross'
        elif 'cross_attention' in k:
            attn_type = 'cross'
        elif 'self_attention' in k:
            attn_type = 'self'

        if 'encoder_decoder' in k:
            scope = 'both'
        elif 'encoder' in k:
            scope = 'encoder'
        elif 'decoder' in k:
            scope = 'decoder'

        rows.append({
            'key': k,
            'scope': scope,
            'attn_type': attn_type,
            'ratio': ratio,
            'Similarity': v['similarity_mean'],
            'ROUGE-L': v['rouge']['rougeL'],
            'METEOR': v['meteor_mean'],
            'Δ Sim': v['similarity_mean'] - baseline_vals[0],
        })
    return pd.DataFrame(rows)

# T5-Large / Dolly
df_attn_dolly = parse_attention_masking(dolly_attn_mask, dolly_bl_vals)
# Flan-T5 base
df_attn_flan = parse_attention_masking(flan_base, flan_bl_vals)

def plot_attention_masking(df, title, ax):
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        return
    for name, grp in df.groupby(['scope', 'attn_type']):
        grp_sorted = grp.sort_values('ratio')
        label = f'{name[0]} / {name[1]}'
        ax.plot(grp_sorted['ratio'], grp_sorted['Similarity'], 'o-', label=label, markersize=5)
    ax.set_xlabel('Masking Ratio')
    ax.set_ylabel('Similarity')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='lower left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_attention_masking(df_attn_dolly, 'T5-Large / Dolly', axes[0])
plot_attention_masking(df_attn_flan, 'Flan-T5 (base) / Self-Instruct', axes[1])
fig.suptitle('Attention Head Masking --- Similarity vs Ratio', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Razvan + Mihnea,  Similarity): The two models behave very differently here, so they must not be lumped together:

- T5-Large / Dolly is catastrophically sensitive to attention masking. Even at the lowest tested ratio (10%) similarity already collapses from the 0.611 baseline to ≈0.30, and it stays in the 0.22-0.36 band up to 75%. For this model, masking attention is almost as destructive as removing whole layers. (Such a cliff already at 10% is steep enough that it is worth confirming the masking runs used the same decoding / baseline setup as the other experiments.)
- Flan-T5 / Self-Instruct is far more robust. At 10% masking similarity is essentially unchanged (0.649-0.665 vs 0.655 baseline), and even at 75% it is still 0.39-0.63. Only here is the "low-ratio masking causes minor degradation" statement true.
- Most damaging configuration (both models): combined self+cross masking on both encoder and decoder — the lowest similarity at 75% (T5 0.217, Flan 0.393), closely followed by decoder self+cross. Decoder cross-attention alone is the most damaging single attention type in T5 (0.223) but not in Flan (0.510).
- Most robust configuration: decoder self-attention, not encoder self-attention (at 75%: T5 decoder-self 0.358 vs encoder-self 0.297; Flan decoder-self 0.634 vs encoder-self 0.560).


In [ ]:
# Attention masking --- ROUGE-L
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [(axes[0], df_attn_dolly, 'T5-Large / Dolly'), (axes[1], df_attn_flan, 'Flan-T5 (base) / Self-Instruct')]:
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    for name, grp in df.groupby(['scope', 'attn_type']):
        grp_sorted = grp.sort_values('ratio')
        label = f'{name[0]} / {name[1]}'
        ax.plot(grp_sorted['ratio'], grp_sorted['ROUGE-L'], 'o-', label=label, markersize=5)
    ax.set_xlabel('Masking Ratio')
    ax.set_ylabel('ROUGE-L')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='lower left')
fig.suptitle('Attention Head Masking --- ROUGE-L vs Ratio', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Attention masking --- METEOR
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, df, title in [(axes[0], df_attn_dolly, 'T5-Large / Dolly'), (axes[1], df_attn_flan, 'Flan-T5 (base) / Self-Instruct')]:
    if len(df) == 0:
        ax.set_title(title + ' (no data)')
        continue
    for name, grp in df.groupby(['scope', 'attn_type']):
        grp_sorted = grp.sort_values('ratio')
        label = f'{name[0]} / {name[1]}'
        ax.plot(grp_sorted['ratio'], grp_sorted['METEOR'], 'o-', label=label, markersize=5)
    ax.set_xlabel('Masking Ratio')
    ax.set_ylabel('METEOR')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='lower left')
fig.suptitle('Attention Head Masking --- METEOR vs Ratio', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 2.2 Feed-Forward Masking

In [ ]:
def parse_ff_masking(results, baseline_vals):
    rows = []
    for k, v in results.items():
        if not k.startswith('mask_') or 'feed_forward' not in k:
            continue
        ratio = float(k.split('_')[-1])
        if 'encoder_feed_forward' in k or k.startswith('mask_encoder_'):
            scope = 'encoder'
        elif 'decoder_feed_forward' in k or k.startswith('mask_decoder_'):
            scope = 'decoder'
        else:
            scope = 'both'
        rows.append({
            'scope': scope, 'ratio': ratio,
            'Similarity': v['similarity_mean'],
            'ROUGE-L': v['rouge']['rougeL'],
            'METEOR': v['meteor_mean'],
        })
    return pd.DataFrame(rows)

df_ffm_dolly = parse_ff_masking(dolly_ff_mask, dolly_bl_vals)
df_ffm_flan = parse_ff_masking(flan_base, flan_bl_vals)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics = [('Similarity', 'Similarity'), ('ROUGE-L', 'ROUGE-L'), ('METEOR', 'METEOR')]
for ax, (col, ylabel) in zip(axes, metrics):
    for scope, marker in [('encoder', 'o'), ('decoder', 's'), ('both', '^')]:
        sub = df_ffm_dolly[df_ffm_dolly['scope'] == scope].sort_values('ratio')
        if len(sub) > 0:
            ax.plot(sub['ratio'], sub[col], f'{marker}-', label=f'T5 {scope}', markersize=5)
    for scope, marker in [('encoder', 'o'), ('decoder', 's'), ('both', '^')]:
        sub = df_ffm_flan[df_ffm_flan['scope'] == scope].sort_values('ratio')
        if len(sub) > 0:
            ax.plot(sub['ratio'], sub[col], f'{marker}--', label=f'Flan {scope}', markersize=5, alpha=0.7)
    ax.set_xlabel('Masking Ratio')
    ax.set_ylabel(ylabel)
    ax.set_title(f'FF Masking --- {ylabel}')
    ax.legend(fontsize=6)
fig.suptitle('Feed-Forward Masking Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Inspect FF masking data at each ratio
print("=== T5-Large / Dolly,  FF Masking ===")
print(df_ffm_dolly.sort_values(['scope','ratio'])[['scope','ratio','Similarity','ROUGE-L','METEOR']].to_string(index=False))
print()
print("=== Flan-T5 Base / Self-Instruct,  FF Masking ===")
print(df_ffm_flan.sort_values(['scope','ratio'])[['scope','ratio','Similarity','ROUGE-L','METEOR']].to_string(index=False))

Findings (Razvan + Mihnea,  FF Masking): FF masking is dramatically more destructive than FF dropping. While dropping entire odd/even FF sub-layers preserved similarity at 0.6+ (see Section 1.6), even 10% random masking collapses similarity to 0.21-0.44. This contrast is critical:

- T5 / Dolly: Encoder FF masking shows a flat degradation curve,  similarity stays at ≈0.21 regardless of masking ratio (10% vs 80%). The encoder FF is so sensitive that any corruption, however small, saturates the damage instantly. Decoder FF masking starts slightly higher (0.28 at 25%) but degrades to the same floor at high ratios.
- Flan-T5 / Self-Instruct: Decoder FF at 10% masking retains more similarity (0.436) than encoder FF (0.373), but both drop sharply at higher ratios. Encoder masking is again nearly flat across ratios.
- ROUGE-L and METEOR are near zero across all FF masking experiments, indicating complete lexical collapse even when some semantic similarity is retained.

The key insight is that structured removal (dropping whole sub-layers) is far more survivable than unstructured corruption (random weight masking), because clean removal allows residual connections to pass information through, while partial masking corrupts the layer's computation while keeping it active.

---
## Part 3: Gaussian Noise Robustness

In [ ]:
def parse_gaussian(results, baseline_vals):
    rows = []
    for k, v in results.items():
        if not k.startswith('add_gaussian_noise'):
            continue
        parts = k.replace('add_gaussian_noise_', '').rsplit('_', 1)
        sigma = float(parts[-1])
        target = parts[0]  # e.g. encoder_all, decoder_feed_forward_all, etc.
        # Simplify
        if 'feed_forward' in target:
            if 'encoder' in target:
                scope = 'encoder FF'
            elif 'decoder' in target:
                scope = 'decoder FF'
            else:
                scope = 'all FF'
        else:
            if 'encoder' in target:
                scope = 'encoder full'
            elif 'decoder' in target:
                scope = 'decoder full'
            else:
                scope = 'all full'
        rows.append({
            'scope': scope, 'sigma': sigma,
            'Similarity': v['similarity_mean'],
            'ROUGE-L': v['rouge']['rougeL'],
            'METEOR': v['meteor_mean'],
        })
    return pd.DataFrame(rows)

df_gauss_dolly = parse_gaussian(dolly_gauss, dolly_bl_vals)
df_gauss_flan = parse_gaussian(flan_base, flan_bl_vals)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, ylabel) in zip(axes, [('Similarity', 'Similarity'), ('ROUGE-L', 'ROUGE-L'), ('METEOR', 'METEOR')]):
    for scope in df_gauss_dolly['scope'].unique():
        sub = df_gauss_dolly[df_gauss_dolly['scope'] == scope].sort_values('sigma')
        ax.plot(sub['sigma'], sub[col], 'o-', label=f'T5 {scope}', markersize=4)
    ax.set_xlabel('Noise σ')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Gaussian Noise --- {ylabel}')
    ax.legend(fontsize=6, loc='lower left')
    ax.set_xscale('log')

fig.suptitle('T5-Large / Dolly --- Gaussian Noise Injection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Razvan,  T5-Large / Dolly): T5-Large is essentially fully robust to Gaussian noise. Across every injection scope and every σ (0.01-1.0) similarity stays in the narrow band 0.610-0.615,  i.e. within ±0.005 of the 0.611 baseline. The per-scope ordering is not meaningful: the differences are smaller than run-to-run noise, so there is no reliable evidence that encoder noise hurts more than FF noise, nor that σ=1.0 encoder noise is the worst (encoder-full at σ=1.0 is actually 0.6125, among the highest values). The only safe conclusion is that additive Gaussian noise at these magnitudes does not perturb T5's outputs.


In [ ]:
# Flan-T5 Gaussian noise
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (col, ylabel) in zip(axes, [('Similarity', 'Similarity'), ('ROUGE-L', 'ROUGE-L'), ('METEOR', 'METEOR')]):
    for scope in sorted(df_gauss_flan['scope'].unique()):
        sub = df_gauss_flan[df_gauss_flan['scope'] == scope].sort_values('sigma')
        ax.plot(sub['sigma'], sub[col], 'o-', label=scope, markersize=4)
    ax.set_xlabel('Noise σ')
    ax.set_ylabel(ylabel)
    ax.set_title(f'Gaussian Noise --- {ylabel}')
    ax.legend(fontsize=6, loc='lower left')
    ax.set_xscale('log')

fig.suptitle('Flan-T5-Large (base) / Self-Instruct --- Gaussian Noise', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Mihnea,  Flan-T5 base / Self-Instruct): The Flan-T5 model is also noise-robust,  similarity stays at 0.649-0.666 across ALL noise levels (σ = 0.01 to 1.0) and ALL scopes. There is essentially zero degradation from Gaussian noise. Encoder-full at σ=0.5 even shows similarity of 0.663, marginally above the 0.655 baseline,  but as with T5 this should be read as noise, not a real improvement. ROUGE-L and METEOR also remain stable. The overall robustness is comparable to T5-Large/Dolly; both models are insensitive to noise at these magnitudes, so this experiment does not, on its own, support a claim that instruction-tuning increases noise tolerance.


In [ ]:
# Encoder full vs Decoder full --- cross-model comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, scope_key, title in [
    (axes[0], 'encoder full', 'Encoder --- Full Noise'),
    (axes[1], 'decoder full', 'Decoder --- Full Noise'),
]:
    sub_d = df_gauss_dolly[df_gauss_dolly['scope'] == scope_key].sort_values('sigma')
    sub_f = df_gauss_flan[df_gauss_flan['scope'] == scope_key].sort_values('sigma')
    if len(sub_d) > 0:
        ax.plot(sub_d['sigma'], sub_d['Similarity'], 'o-', label='T5 Sim', markersize=5)
        ax.plot(sub_d['sigma'], sub_d['ROUGE-L'], 's-', label='T5 ROUGE-L', markersize=5)
    if len(sub_f) > 0:
        ax.plot(sub_f['sigma'], sub_f['Similarity'], 'o--', label='Flan Sim', markersize=5, alpha=0.7)
        ax.plot(sub_f['sigma'], sub_f['ROUGE-L'], 's--', label='Flan ROUGE-L', markersize=5, alpha=0.7)
    ax.set_xlabel('Noise σ')
    ax.set_ylabel('Score')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xscale('log')

fig.suptitle('Gaussian Noise --- Cross-Model Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Flan-T5 Student v1 Comparison
Same experiments, comparing base (sim baseline=0.655) vs student distilled model (sim baseline=0.590).

In [ ]:
# Base vs Student: single encoder drop
flan_stud_bl = flan_student['baseline_no_drop']

enc_base, enc_stud = [], []
for i in range(24):
    k = f'drop_encoder_block_{i}'
    if k in flan_base:
        v = flan_base[k]
        enc_base.append({'layer': i, 'delta_sim': v['similarity_mean'] - flan_base_bl['similarity_mean']})
    if k in flan_student:
        v = flan_student[k]
        enc_stud.append({'layer': i, 'delta_sim': v['similarity_mean'] - flan_stud_bl['similarity_mean']})

df_eb = pd.DataFrame(enc_base)
df_es = pd.DataFrame(enc_stud)

fig, ax = plt.subplots(figsize=(12, 4))
if len(df_eb) > 0:
    ax.plot(df_eb['layer'], df_eb['delta_sim'], 'o-', label='Flan-T5 Base', markersize=4)
if len(df_es) > 0:
    ax.plot(df_es['layer'], df_es['delta_sim'], 's-', label='Flan-T5 Student', markersize=4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax.set_xlabel('Encoder Layer Index')
ax.set_ylabel('Δ Similarity')
ax.set_title('Base vs Student --- Single Encoder Layer Drop (Δ Similarity)')
ax.legend()
plt.tight_layout()
plt.show()

Findings (Mihnea,  Base vs Student, Encoder): The student model shows larger per-layer drops than the base model. The student's worst encoder layers are layer 23 (Δ sim = −0.052) and layer 0 (Δ sim = −0.052), compared to base layer 23 (Δ sim = −0.041). Notably, encoder layer 0 becomes critical in the student (Δ = −0.052) while in the base it's essentially harmless (Δ ≈ −0.003). Distillation has made the encoder's edge layers more important.

In [ ]:
# Base vs Student: single decoder drop
dec_base, dec_stud = [], []
for i in range(24):
    k = f'drop_decoder_block_{i}'
    if k in flan_base:
        v = flan_base[k]
        dec_base.append({'layer': i, 'delta_sim': v['similarity_mean'] - flan_base_bl['similarity_mean']})
    if k in flan_student:
        v = flan_student[k]
        dec_stud.append({'layer': i, 'delta_sim': v['similarity_mean'] - flan_stud_bl['similarity_mean']})

df_db = pd.DataFrame(dec_base)
df_ds = pd.DataFrame(dec_stud)

fig, ax = plt.subplots(figsize=(12, 4))
if len(df_db) > 0:
    ax.plot(df_db['layer'], df_db['delta_sim'], 'o-', label='Flan-T5 Base', markersize=4)
if len(df_ds) > 0:
    ax.plot(df_ds['layer'], df_ds['delta_sim'], 's-', label='Flan-T5 Student', markersize=4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.7)
ax.set_xlabel('Decoder Layer Index')
ax.set_ylabel('Δ Similarity')
ax.set_title('Base vs Student --- Single Decoder Layer Drop (Δ Similarity)')
ax.legend()
plt.tight_layout()
plt.show()

Findings (Mihnea,  Base vs Student, Decoder): Both models share decoder layer 6 as the critical layer, but the student is actually more robust there: student Δ sim = −0.183 vs base Δ sim = −0.265 (30% less damage). Distillation appears to have partially redistributed the decoder layer 6 dependency, spreading it across other layers.

In [ ]:
# Base vs Student: Gaussian noise (similarity on encoder full)
df_gauss_stud = parse_gaussian(flan_student, (flan_stud_bl['similarity_mean'], flan_stud_bl['rouge']['rougeL'], flan_stud_bl['meteor_mean']))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, scope_key, title in [
    (axes[0], 'encoder full', 'Encoder Full Noise'),
    (axes[1], 'decoder full', 'Decoder Full Noise'),
]:
    sub_b = df_gauss_flan[df_gauss_flan['scope'] == scope_key].sort_values('sigma')
    sub_s = df_gauss_stud[df_gauss_stud['scope'] == scope_key].sort_values('sigma')
    if len(sub_b) > 0:
        ax.plot(sub_b['sigma'], sub_b['Similarity'], 'o-', label='Base Sim', markersize=5)
    if len(sub_s) > 0:
        ax.plot(sub_s['sigma'], sub_s['Similarity'], 's--', label='Student Sim', markersize=5)
    ax.set_xlabel('Noise σ')
    ax.set_ylabel('Similarity')
    ax.set_title(title)
    ax.legend()
    ax.set_xscale('log')

fig.suptitle('Flan-T5 Base vs Student --- Gaussian Noise Robustness', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Base vs Student: attention masking comparison
df_attn_stud = parse_attention_masking(flan_student, (flan_stud_bl['similarity_mean'], flan_stud_bl['rouge']['rougeL'], flan_stud_bl['meteor_mean']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_attention_masking(df_attn_flan, 'Flan-T5 Base', axes[0])
plot_attention_masking(df_attn_stud, 'Flan-T5 Student', axes[1])
fig.suptitle('Attention Masking --- Base vs Student', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Mihnea,  Base vs Student, Attention Masking): The student model is consistently more sensitive to attention masking at 75% ratio than the base. Key differences:
- Decoder cross-attention: student 0.379 vs base 0.510,  student loses 25% more similarity
- Encoder self-attention: student 0.484 vs base 0.560
- Combined self+cross on both: student 0.374 vs base 0.393

Distillation reduces attention head redundancy,  the student relies more heavily on each remaining head, making masking more damaging. This contrasts with decoder layer 6, where distillation actually improved robustness.

In [ ]:
# Inspect Flan Gaussian + cross-model + base vs student data
print("=== Flan-T5 Gaussian,  all scopes ===")
print(df_gauss_flan.sort_values(['scope','sigma'])[['scope','sigma','Similarity','ROUGE-L','METEOR']].to_string(index=False))
print()

# Base vs Student: worst encoder/decoder layers
print("=== Base vs Student: Encoder worst 5 ===")
print("Base:", pd.DataFrame(enc_base).sort_values('delta_sim').head(5)[['layer','delta_sim']].to_string(index=False))
print("Student:", pd.DataFrame(enc_stud).sort_values('delta_sim').head(5)[['layer','delta_sim']].to_string(index=False))
print()
print("=== Base vs Student: Decoder worst 5 ===")
print("Base:", pd.DataFrame(dec_base).sort_values('delta_sim').head(5)[['layer','delta_sim']].to_string(index=False))
print("Student:", pd.DataFrame(dec_stud).sort_values('delta_sim').head(5)[['layer','delta_sim']].to_string(index=False))
print()

# Gaussian student data
print("=== Student Gaussian,  encoder/decoder full ===")
for scope_key in ['encoder full', 'decoder full']:
    sub = df_gauss_stud[df_gauss_stud['scope'] == scope_key].sort_values('sigma')
    print(f"\n{scope_key}:")
    print(sub[['sigma','Similarity']].to_string(index=False))

# Attention student vs base at 75% masking
print("\n=== Attention masking at 75%,  Base vs Student ===")
for df_name, df_data, label in [('Base', df_attn_flan, 'Base'), ('Student', df_attn_stud, 'Student')]:
    sub = df_data[df_data['ratio'] == 0.75]
    if len(sub) > 0:
        print(f"\n{label}:")
        print(sub[['scope','attn_type','Similarity']].to_string(index=False))

---
## Summary Heatmaps

In [ ]:
# Heatmap: all single-layer drops --- similarity delta
# Rows = layer index, Cols = encoder/decoder for each model

n_layers = 24
heat_data = np.full((n_layers, 4), np.nan)
col_labels = ['T5 Enc', 'T5 Dec', 'Flan Enc', 'Flan Dec']

for i in range(n_layers):
    k = f'drop_encoder_block_{i}'
    if k in dolly_component:
        heat_data[i, 0] = dolly_component[k]['similarity_mean'] - dolly_bl_sim
    if k in flan_base:
        heat_data[i, 2] = flan_base[k]['similarity_mean'] - flan_base_bl['similarity_mean']
    k = f'drop_decoder_block_{i}'
    if k in dolly_component:
        heat_data[i, 1] = dolly_component[k]['similarity_mean'] - dolly_bl_sim
    if k in flan_base:
        heat_data[i, 3] = flan_base[k]['similarity_mean'] - flan_base_bl['similarity_mean']

fig, ax = plt.subplots(figsize=(6, 10))
im = ax.imshow(heat_data, cmap='RdYlGn', aspect='auto', vmin=-0.25, vmax=0.05)
ax.set_xticks(range(4))
ax.set_xticklabels(col_labels, fontsize=9)
ax.set_yticks(range(n_layers))
ax.set_yticklabels(range(n_layers))
ax.set_ylabel('Layer Index')
ax.set_title('Δ Similarity --- Single Layer Drop')
plt.colorbar(im, ax=ax, shrink=0.6)
plt.tight_layout()
plt.show()

Findings (Heatmap): The heatmap clearly visualizes the two critical spikes: T5 decoder layer 0 (yellow/orange cell at row 0, column "T5 Dec") and Flan decoder layer 6 (bright red cell at row 6, column "Flan Dec"). The Flan decoder layer 6 spike (Δ ≈ −0.265) is visibly deeper red than the T5 decoder layer 0 spike (Δ ≈ −0.137). Both encoder columns are uniformly green (near-zero Δ), confirming the encoder's broad redundancy. The Flan encoder column shows a slightly lighter shade at layer 23 (Δ ≈ −0.041), the only notable encoder sensitivity point.

In [ ]:
# Summary table: Gaussian noise at sigma=0.5 for all scopes
print('=== Gaussian Noise σ=0.5 --- T5-Large / Dolly ===')
sub = df_gauss_dolly[df_gauss_dolly['sigma'] == 0.5][['scope', 'Similarity', 'ROUGE-L', 'METEOR']]
print(sub.to_string(index=False))
print()
print('=== Gaussian Noise σ=0.5 --- Flan-T5 Base / Self-Instruct ===')
sub = df_gauss_flan[df_gauss_flan['sigma'] == 0.5][['scope', 'Similarity', 'ROUGE-L', 'METEOR']]
print(sub.to_string(index=False))

In [ ]:
# Final: three-metric scatter for all experiments (Flan base)
all_rows = []
for k, v in flan_base.items():
    if k == 'baseline_no_drop':
        continue
    all_rows.append({
        'key': k,
        'Similarity': v['similarity_mean'],
        'ROUGE-L': v['rouge']['rougeL'],
        'METEOR': v['meteor_mean'],
    })
df_all = pd.DataFrame(all_rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].scatter(df_all['Similarity'], df_all['ROUGE-L'], s=10, alpha=0.5)
axes[0].set_xlabel('Similarity')
axes[0].set_ylabel('ROUGE-L')
axes[0].set_title('Similarity vs ROUGE-L')

axes[1].scatter(df_all['Similarity'], df_all['METEOR'], s=10, alpha=0.5, color='orange')
axes[1].set_xlabel('Similarity')
axes[1].set_ylabel('METEOR')
axes[1].set_title('Similarity vs METEOR')

axes[2].scatter(df_all['ROUGE-L'], df_all['METEOR'], s=10, alpha=0.5, color='green')
axes[2].set_xlabel('ROUGE-L')
axes[2].set_ylabel('METEOR')
axes[2].set_title('ROUGE-L vs METEOR')

fig.suptitle('Metric Correlations --- All Ablations (Flan-T5 base)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

Findings (Metric Correlations): The scatter plots reveal a clear two-cluster structure:
- A high cluster (sim ≈ 0.55-0.67, ROUGE-L ≈ 0.15-0.30, METEOR ≈ 0.10-0.20): experiments with mild ablations (single-layer drops, attention masking at low ratios, Gaussian noise). These preserve both semantic and lexical similarity.
- A low cluster (sim ≈ 0.25-0.45, ROUGE-L ≈ 0.00-0.05, METEOR ≈ 0.00-0.05): experiments with severe ablations (FF masking, cumulative drops, full component removal). Here semantic similarity partially survives (0.25-0.45) while lexical metrics collapse to near zero.

This gap between the two clusters shows that ROUGE-L and METEOR are much more fragile than embedding similarity, a model can preserve meaning while completely changing its wording. The strong ROUGE-L vs METEOR correlation (bottom-right panel) confirms these two lexical metrics track each other closely.